In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [2]:
df = pd.read_excel(r"C:\Users\hp\OneDrive\Desktop\Retail-Analytics-Platform\Data\clean_raw.xlsx")

# Customer table

In [3]:
customers = df[["Customer ID","Customer Name", "Segment"]]

In [4]:
customers = customers.drop_duplicates()

In [5]:
print(customers["Customer ID"].is_unique)

True


In [6]:
customers.reset_index(drop=True, inplace=True)

In [7]:
customers.shape

(17415, 3)

In [8]:
customers.to_csv(r"C:\Users\hp\OneDrive\Desktop\Retail-Analytics-Platform\Data\ETL data\Customers.csv",index=False)

In [9]:
data = pd.read_csv(r"C:\Users\hp\OneDrive\Desktop\Retail-Analytics-Platform\Data\ETL data\Customers.csv")

In [10]:
data.head(3)

,Customer ID,Customer Name,Segment
0,AB-100151402,Aaron Bergman,Consumer
1,JR-162107,Justin Ritter,Corporate
2,CR-127307,Craig Reiter,Consumer


In [11]:
data.shape

(17415, 3)

In [12]:
data.duplicated().sum()

np.int64(0)

In [13]:
data["Customer ID"].is_unique

True

In [14]:
customer_issues = data.groupby("Customer ID")["Customer Name"].nunique()
customer_issues = customer_issues[customer_issues > 1]
print(customer_issues)

Series([], Name: Customer Name, dtype: int64)


In [15]:
customer_issues = data.groupby("Customer ID")["Segment"].nunique()
customer_issues = customer_issues[customer_issues > 1]
print(customer_issues)

Series([], Name: Segment, dtype: int64)


#### Validation function

In [16]:
def validate_one_to_one(df, key, value):
    """
    Validates that every key maps to exactly one value.
    """
    issues = (df.groupby(key)[value].nunique())
    issues = issues[issues > 1]
    if len(issues) == 0:
        print(f"PASS: {key} uniquely maps to {value}")
    else:
        print(f"FAIL: {len(issues)} inconsistencies found")
        return issues

In [17]:
validate_one_to_one(data, "Customer ID", "Customer Name")
validate_one_to_one(data, "Customer ID", "Segment")

PASS: Customer ID uniquely maps to Customer Name
PASS: Customer ID uniquely maps to Segment


# Product table

In [18]:
products = df[["Product ID","Product Name","Category", "Sub-Category"]]

In [19]:
products = products.drop_duplicates()

In [20]:
products.reset_index(drop=True, inplace=True)

In [21]:
products["Product ID"].is_unique

True

#### Validation

In [22]:
validate_one_to_one(products,"Product ID","Product Name")

PASS: Product ID uniquely maps to Product Name


In [23]:
validate_one_to_one(products,"Product ID","Category")

PASS: Product ID uniquely maps to Category


In [24]:
validate_one_to_one(products,"Product ID","Sub-Category")

PASS: Product ID uniquely maps to Sub-Category


In [25]:
products.to_csv(r"C:\Users\hp\OneDrive\Desktop\Retail-Analytics-Platform\Data\ETL data\Products.csv",index=False)

## Investigation about columns

In [26]:
order_level_columns = ["Customer ID","Order Date","Ship Date","Ship Mode","Order Priority","Postal Code","City","State","Country","Region","Market",
                       "Shipping Cost"]

In [27]:
for column in order_level_columns:
    result = (df.groupby("Order ID")[column].nunique(dropna=False))
    inconsistent = result[result > 1]
    print(f"{column}: {len(inconsistent)} inconsistent orders")

Customer ID: 11 inconsistent orders
Order Date: 0 inconsistent orders
Ship Date: 21 inconsistent orders
Ship Mode: 17 inconsistent orders
Order Priority: 17 inconsistent orders
Postal Code: 11 inconsistent orders
City: 25 inconsistent orders
State: 21 inconsistent orders
Country: 0 inconsistent orders
Region: 11 inconsistent orders
Market: 0 inconsistent orders
Shipping Cost: 12786 inconsistent orders


In [28]:
shipping_check = (df.groupby("Order ID")["Shipping Cost"].nunique(dropna=False))
shipping_inconsistent = shipping_check[shipping_check > 1]
print("Orders with multiple shipping costs:",
      len(shipping_inconsistent))

Orders with multiple shipping costs: 12786


In [29]:
unique_orders = df["Order ID"].nunique()
print("Unique Orders:", unique_orders)

Unique Orders: 25728


In [30]:
print("Total Rows:", len(df))

Total Rows: 51290


#### Validation

In [31]:
customer_check = (df.groupby("Order ID")["Customer ID"].nunique(dropna=False))
customer_check[customer_check > 1].head()

Order ID
CA-2012-CS12355140-41135    2
CA-2013-EM13960140-41548    2
CA-2013-KB16240140-41585    2
CA-2013-PJ18835140-41592    2
CA-2014-BW11200140-41940    2
Name: Customer ID, dtype: int64

In [32]:
problem_order = customer_check[customer_check > 1].index[0]
df[df["Order ID"] == problem_order][["Order ID","Customer ID","Customer Name","Order Date","Ship Date","Ship Mode","Order Priority","Postal Code",
                                     "City","State","Country","Region","Market","Shipping Cost"]]

,Order ID,Customer ID,Customer Name,Order Date,Ship Date,Ship Mode,Order Priority,Postal Code,City,State,Country,Region,Market,Shipping Cost
10914,CA-2012-CS12355140-41135,CS-123551408,Christine Sundaresam,2012-08-14,2012-08-16,First Class,High,33021.0,Hollywood,Florida,United States,Southern US,USCA,26.77
10934,CA-2012-CS12355140-41135,CS-123551408,Christine Sundaresam,2012-08-14,2012-08-16,First Class,High,33021.0,Hollywood,Florida,United States,Southern US,USCA,14.78
10947,CA-2012-CS12355140-41135,CS-123551406,Christine Sundaresam,2012-08-14,2012-08-17,First Class,High,6824.0,Fairfield,Connecticut,United States,Eastern US,USCA,9.61


In [33]:
ship_date_check = (df.groupby("Order ID")["Ship Date"].nunique(dropna=False))
ship_date_check[ship_date_check > 1].head()

Order ID
CA-2012-CS12355140-41135    2
CA-2013-EM13960140-41548    2
CA-2013-KB16240140-41585    2
CA-2013-PJ18835140-41592    2
CA-2014-BW11200140-41940    2
Name: Ship Date, dtype: int64

In [34]:
problem_order = ship_date_check[ship_date_check > 1].index[0]
df[df["Order ID"] == problem_order][["Order ID", "Order Date", "Ship Date", "Ship Mode"]]

,Order ID,Order Date,Ship Date,Ship Mode
10914,CA-2012-CS12355140-41135,2012-08-14,2012-08-16,First Class
10934,CA-2012-CS12355140-41135,2012-08-14,2012-08-16,First Class
10947,CA-2012-CS12355140-41135,2012-08-14,2012-08-17,First Class


In [35]:
city_check = (df.groupby("Order ID")["City"].nunique(dropna=False))
city_check[city_check > 1].head()

Order ID
CA-2012-CS12355140-41135    2
CA-2013-EM13960140-41548    2
CA-2013-KB16240140-41585    2
CA-2013-PJ18835140-41592    2
CA-2014-BW11200140-41940    2
Name: City, dtype: int64

In [36]:
shipping_check = (df.groupby("Order ID")["Shipping Cost"].nunique(dropna=False))
shipping_inconsistent = shipping_check[shipping_check > 1]
shipping_inconsistent.head()

Order ID
AE-2012-PO8865138-41184    2
AE-2014-EB4110138-41926    2
AE-2014-MY7380138-42004    2
AE-2015-GH4665138-42351    5
AG-2012-BM17853-41085      3
Name: Shipping Cost, dtype: int64

In [37]:
problem_order = shipping_inconsistent.index[0]
df[df["Order ID"] == problem_order][["Order ID","Product ID","Product Name","Quantity","Sales","Shipping Cost"]]

,Order ID,Product ID,Product Name,Quantity,Sales,Shipping Cost
29575,AE-2012-PO8865138-41184,OFF-ST-4258,"Fellowes File Cart, Industrial",2,82.674,5.69
34130,AE-2012-PO8865138-41184,TEC-MA-4190,"Epson Calculator, Red",6,78.408,3.87


In [38]:
customer_check[customer_check > 1].head()

Order ID
CA-2012-CS12355140-41135    2
CA-2013-EM13960140-41548    2
CA-2013-KB16240140-41585    2
CA-2013-PJ18835140-41592    2
CA-2014-BW11200140-41940    2
Name: Customer ID, dtype: int64

In [39]:
shipping_inconsistent.head()

Order ID
AE-2012-PO8865138-41184    2
AE-2014-EB4110138-41926    2
AE-2014-MY7380138-42004    2
AE-2015-GH4665138-42351    5
AG-2012-BM17853-41085      3
Name: Shipping Cost, dtype: int64

In [40]:
df[df["Order ID"] == problem_order][["Order ID", "Product ID", "Product Name", "Quantity", "Sales", "Shipping Cost"]]

,Order ID,Product ID,Product Name,Quantity,Sales,Shipping Cost
29575,AE-2012-PO8865138-41184,OFF-ST-4258,"Fellowes File Cart, Industrial",2,82.674,5.69
34130,AE-2012-PO8865138-41184,TEC-MA-4190,"Epson Calculator, Red",6,78.408,3.87


In [41]:
customer_problem_order = customer_check[customer_check > 1].index[0]
df[df["Order ID"] == customer_problem_order][["Order ID","Customer ID","Customer Name","Segment","Order Date","Product ID","Product Name"]]

,Order ID,Customer ID,Customer Name,Segment,Order Date,Product ID,Product Name
10914,CA-2012-CS12355140-41135,CS-123551408,Christine Sundaresam,Consumer,2012-08-14,OFF-AP-4292,Fellowes Superior 10 Outlet Split Surge Protector
10934,CA-2012-CS12355140-41135,CS-123551408,Christine Sundaresam,Consumer,2012-08-14,OFF-PA-6511,Xerox 1955
10947,CA-2012-CS12355140-41135,CS-123551406,Christine Sundaresam,Consumer,2012-08-14,OFF-BI-5637,Premium Transparent Presentation Covers by GBC


In [42]:
customer_issues = (df.groupby("Order ID")["Customer ID"].nunique(dropna=False))
customer_issues = customer_issues[customer_issues > 1]
print("Problematic orders:", len(customer_issues))

Problematic orders: 11


In [43]:
df[df["Order ID"].isin(customer_issues.index)][["Order ID","Customer ID","Customer Name","Order Date"]].sort_values("Order ID")

,Order ID,Customer ID,Customer Name,Order Date
10914,CA-2012-CS12355140-41135,CS-123551408,Christine Sundaresam,2012-08-14
10934,CA-2012-CS12355140-41135,CS-123551408,Christine Sundaresam,2012-08-14
10947,CA-2012-CS12355140-41135,CS-123551406,Christine Sundaresam,2012-08-14
19893,CA-2013-EM13960140-41548,EM-139601404,Eric Murdock,2013-10-01
19909,CA-2013-EM13960140-41548,EM-139601402,Eric Murdock,2013-10-01
19933,CA-2013-EM13960140-41548,EM-139601402,Eric Murdock,2013-10-01
31371,CA-2013-KB16240140-41585,KB-162401404,Karen Bern,2013-11-07
31395,CA-2013-KB16240140-41585,KB-162401406,Karen Bern,2013-11-07
42715,CA-2013-PJ18835140-41592,PJ-188351408,Patrick Jones,2013-11-14
42717,CA-2013-PJ18835140-41592,PJ-188351408,Patrick Jones,2013-11-14


In [44]:
ids = ["CS-123551408","CS-123551406"]
df[df["Customer ID"].isin(ids)][["Customer ID","Customer Name","Segment","Order ID"]].sort_values("Customer ID")

,Customer ID,Customer Name,Segment,Order ID
10917,CS-123551406,Christine Sundaresam,Consumer,CA-2015-CS12355140-42172
10947,CS-123551406,Christine Sundaresam,Consumer,CA-2012-CS12355140-41135
10961,CS-123551406,Christine Sundaresam,Consumer,CA-2015-CS12355140-42109
10982,CS-123551406,Christine Sundaresam,Consumer,CA-2015-CS12355140-42193
10984,CS-123551406,Christine Sundaresam,Consumer,CA-2015-CS12355140-42172
10986,CS-123551406,Christine Sundaresam,Consumer,CA-2014-CS12355140-41716
10987,CS-123551406,Christine Sundaresam,Consumer,CA-2015-CS12355140-42109
10898,CS-123551408,Christine Sundaresam,Consumer,CA-2013-CS12355140-41521
10900,CS-123551408,Christine Sundaresam,Consumer,CA-2013-CS12355140-41521
10902,CS-123551408,Christine Sundaresam,Consumer,CA-2015-CS12355140-42363


In [45]:
customers[customers["Customer ID"].isin(ids)]

,Customer ID,Customer Name,Segment
6597,CS-123551408,Christine Sundaresam,Consumer
6607,CS-123551406,Christine Sundaresam,Consumer


In [46]:
df.groupby("Customer ID")["Customer Name"].nunique().loc[lambda x: x.index.isin(ids)]

Customer ID
CS-123551406    1
CS-123551408    1
Name: Customer Name, dtype: int64

In [47]:
df[df["Customer ID"].isin(ids)].groupby("Customer ID")["Customer Name"].unique()

Customer ID
CS-123551406    [Christine Sundaresam]
CS-123551408    [Christine Sundaresam]
Name: Customer Name, dtype: object

In [48]:
for order_id in customer_issues.index:
    temp = df[df["Order ID"] == order_id]
    print("\nOrder:", order_id)
    print(temp[["Customer ID", "Customer Name"]].drop_duplicates().to_string(index=False))


Order: CA-2012-CS12355140-41135
 Customer ID        Customer Name
CS-123551408 Christine Sundaresam
CS-123551406 Christine Sundaresam

Order: CA-2013-EM13960140-41548
 Customer ID Customer Name
EM-139601404  Eric Murdock
EM-139601402  Eric Murdock

Order: CA-2013-KB16240140-41585
 Customer ID Customer Name
KB-162401404    Karen Bern
KB-162401406    Karen Bern

Order: CA-2013-PJ18835140-41592
 Customer ID Customer Name
PJ-188351408 Patrick Jones
PJ-188351406 Patrick Jones

Order: CA-2014-BW11200140-41940
 Customer ID Customer Name
BW-112001406   Ben Wallace
BW-112001408   Ben Wallace

Order: CA-2014-GT14710140-41689
 Customer ID Customer Name
GT-147101406     Greg Tran
GT-147101404     Greg Tran

Order: CA-2014-JE15715140-41999
 Customer ID Customer Name
JE-157151406    Joe Elijah
JE-157151404    Joe Elijah

Order: CA-2014-JG15160140-41979
 Customer ID Customer Name
JG-151601408  James Galang
JG-151601406  James Galang

Order: CA-2015-AG10330140-42361
 Customer ID Customer Name
AG-1033

In [49]:
ids = ["CS-123551406","CS-123551408"]
df[df["Customer ID"].isin(ids)].groupby("Customer ID").agg({
    "Customer Name": "unique",
    "Segment": "unique",
    "City": "unique",
    "State": "unique",
    "Country": "unique",
    "Region": "unique",
    "Market": "unique"
})

,Customer Name,Segment,City,State,Country,Region,Market
Customer ID,,,,,,,
CS-123551406,[Christine Sundaresam],[Consumer],"[Dublin, Fairfield, New York City, Long Beach,...","[Ohio, Connecticut, New York, Massachusetts]",[United States],[Eastern US],[USCA]
CS-123551408,[Christine Sundaresam],[Consumer],"[Roswell, Lafayette, Hollywood, Salem]","[Georgia, Louisiana, Florida, Virginia]",[United States],[Southern US],[USCA]


In [50]:
problem_rows = df[df["Order ID"].isin(customer_issues.index)]
customer_profile_check = (problem_rows.groupby("Customer ID").agg({
        "Customer Name": "unique",
        "Segment": "unique",
        "City": "unique",
        "State": "unique",
        "Country": "unique",
        "Region": "unique",
        "Market": "unique"
    }))
customer_profile_check

,Customer Name,Segment,City,State,Country,Region,Market
Customer ID,,,,,,,
AG-103301402,[Alex Grayson],[Consumer],[Houston],[Texas],[United States],[Central US],[USCA]
AG-103301404,[Alex Grayson],[Consumer],[Mesa],[Arizona],[United States],[Western US],[USCA]
BW-112001406,[Ben Wallace],[Consumer],[New York City],[New York],[United States],[Eastern US],[USCA]
BW-112001408,[Ben Wallace],[Consumer],[Hampton],[Virginia],[United States],[Southern US],[USCA]
CS-123551406,[Christine Sundaresam],[Consumer],[Fairfield],[Connecticut],[United States],[Eastern US],[USCA]
CS-123551408,[Christine Sundaresam],[Consumer],[Hollywood],[Florida],[United States],[Southern US],[USCA]
EM-139601402,[Eric Murdock],[Consumer],[Quincy],[Illinois],[United States],[Central US],[USCA]
EM-139601404,[Eric Murdock],[Consumer],[Portland],[Oregon],[United States],[Western US],[USCA]
GT-147101404,[Greg Tran],[Consumer],[San Francisco],[California],[United States],[Western US],[USCA]


### Customer ID conflicts → quarantined
#### Creating exception table

In [51]:
order_customer_exceptions = df[df["Order ID"].isin(customer_issues.index)][[
        "Order ID",
        "Customer ID",
        "Customer Name",
        "Segment",
        "Order Date",
        "City",
        "State",
        "Country"
    ]].drop_duplicates()

In [52]:
order_customer_exceptions.shape

(22, 8)

In [53]:
order_customer_exceptions.sort_values("Order ID")

,Order ID,Customer ID,Customer Name,Segment,Order Date,City,State,Country
10914,CA-2012-CS12355140-41135,CS-123551408,Christine Sundaresam,Consumer,2012-08-14,Hollywood,Florida,United States
10947,CA-2012-CS12355140-41135,CS-123551406,Christine Sundaresam,Consumer,2012-08-14,Fairfield,Connecticut,United States
19893,CA-2013-EM13960140-41548,EM-139601404,Eric Murdock,Consumer,2013-10-01,Portland,Oregon,United States
19909,CA-2013-EM13960140-41548,EM-139601402,Eric Murdock,Consumer,2013-10-01,Quincy,Illinois,United States
31371,CA-2013-KB16240140-41585,KB-162401404,Karen Bern,Corporate,2013-11-07,Los Angeles,California,United States
31395,CA-2013-KB16240140-41585,KB-162401406,Karen Bern,Corporate,2013-11-07,Philadelphia,Pennsylvania,United States
42715,CA-2013-PJ18835140-41592,PJ-188351408,Patrick Jones,Corporate,2013-11-14,Springfield,Virginia,United States
42722,CA-2013-PJ18835140-41592,PJ-188351406,Patrick Jones,Corporate,2013-11-14,New York City,New York,United States
5741,CA-2014-BW11200140-41940,BW-112001406,Ben Wallace,Consumer,2014-10-28,New York City,New York,United States
5752,CA-2014-BW11200140-41940,BW-112001408,Ben Wallace,Consumer,2014-10-28,Hampton,Virginia,United States


In [54]:
order_customer_exceptions.to_csv(r'C:\Users\hp\OneDrive\Desktop\Retail-Analytics-Platform\Data\ETL data\Exceptions.xls', index=False)

#### Creating valid dataset

In [55]:
valid_orders_data = df[~df["Order ID"].isin(customer_issues.index)].copy()

In [56]:
print("Original rows:", len(df))
print("Valid rows:", len(valid_orders_data))
print("Exception rows:", len(order_customer_exceptions))

Original rows: 51290
Valid rows: 51255
Exception rows: 22


### Validation of remaining columns

In [57]:
for column in ["Ship Date","Ship Mode","Order Priority","Postal Code","City","State","Region"]:  
    check = (valid_orders_data.groupby("Order ID")[column].nunique())  
    inconsistent = check[check > 1]
    print(f"{column}: {len(inconsistent)} true conflicts")

Ship Date: 10 true conflicts
Ship Mode: 9 true conflicts
Order Priority: 11 true conflicts
Postal Code: 0 true conflicts
City: 14 true conflicts
State: 10 true conflicts
Region: 0 true conflicts


In [58]:
# SHIP DATE
ship_date_check = (valid_orders_data.groupby("Order ID")["Ship Date"].nunique())
ship_date_conflicts = ship_date_check[ship_date_check > 1]
print("Conflicting orders:", len(ship_date_conflicts))
print(ship_date_conflicts.head())

Conflicting orders: 10
Order ID
ES-2013-CD1192045-41609     2
ES-2015-BG1174045-42353     2
ES-2015-TM21490139-42223    2
IN-2013-DM133457-41306      2
IN-2015-RF1984058-42273     2
Name: Ship Date, dtype: int64


In [59]:
order = ship_date_conflicts.index[0]
valid_orders_data[valid_orders_data["Order ID"] == order][[
        "Order ID",
        "Order Date",
        "Ship Date",
        "Ship Mode",
        "Product ID",
        "Product Name"
    ]]

,Order ID,Order Date,Ship Date,Ship Mode,Product ID,Product Name
1208,ES-2013-CD1192045-41609,2013-12-01,2013-12-05,Standard Class,TEC-CO-6011,"Sharp Wireless Fax, High-Speed"
7640,ES-2013-CD1192045-41609,2013-12-01,2013-12-07,Standard Class,OFF-ST-4057,"Eldon File Cart, Single Width"
9282,ES-2013-CD1192045-41609,2013-12-01,2013-12-07,Standard Class,FUR-CH-4545,"Harbour Creations Rocking Chair, Black"
26934,ES-2013-CD1192045-41609,2013-12-01,2013-12-05,Standard Class,OFF-BI-3247,"Avery Binder Covers, Durable"
36022,ES-2013-CD1192045-41609,2013-12-01,2013-12-07,Standard Class,OFF-BI-3723,"Cardinal Binder, Recycled"
40439,ES-2013-CD1192045-41609,2013-12-01,2013-12-07,Standard Class,OFF-AR-3452,"BIC Highlighters, Easy-Erase"
42867,ES-2013-CD1192045-41609,2013-12-01,2013-12-07,Standard Class,OFF-BI-2891,"Acco Binder Covers, Clear"
49743,ES-2013-CD1192045-41609,2013-12-01,2013-12-05,Standard Class,OFF-LA-4687,"Hon Round Labels, Alphabetical"


In [60]:
# SHIP MODE
ship_mode_check = (valid_orders_data.groupby("Order ID")["Ship Mode"].nunique())
ship_mode_conflicts = ship_mode_check[ship_mode_check > 1]
print("Conflicting orders:", len(ship_mode_conflicts))

Conflicting orders: 9


In [61]:
order = ship_mode_conflicts.index[0]
valid_orders_data[valid_orders_data["Order ID"] == order][[
        "Order ID",
        "Order Date",
        "Ship Date",
        "Ship Mode",
        "Product ID",
        "Product Name" ]]

,Order ID,Order Date,Ship Date,Ship Mode,Product ID,Product Name
9739,ES-2013-JF1541545-41509,2013-08-23,2013-08-27,Second Class,FUR-BO-3639,"Bush Library with Doors, Metal"
13669,ES-2013-JF1541545-41509,2013-08-23,2013-08-27,Standard Class,FUR-CH-5751,"SAFCO Bag Chairs, Black"
33129,ES-2013-JF1541545-41509,2013-08-23,2013-08-27,Second Class,OFF-AR-6115,"Stanley Markers, Easy-Erase"


In [62]:
# ORDER PRIORITY
priority_check = (valid_orders_data.groupby("Order ID")["Order Priority"].nunique())
priority_conflicts = priority_check[priority_check > 1]
print("Conflicting orders:", len(priority_conflicts))

Conflicting orders: 11


In [63]:
order = priority_conflicts.index[0]
valid_orders_data[valid_orders_data["Order ID"] == order][[
        "Order ID",
        "Order Date",
        "Order Priority",
        "Ship Date",
        "Ship Mode",
        "Product ID",
        "Product Name"]]

,Order ID,Order Date,Order Priority,Ship Date,Ship Mode,Product ID,Product Name
1208,ES-2013-CD1192045-41609,2013-12-01,High,2013-12-05,Standard Class,TEC-CO-6011,"Sharp Wireless Fax, High-Speed"
7640,ES-2013-CD1192045-41609,2013-12-01,Medium,2013-12-07,Standard Class,OFF-ST-4057,"Eldon File Cart, Single Width"
9282,ES-2013-CD1192045-41609,2013-12-01,Medium,2013-12-07,Standard Class,FUR-CH-4545,"Harbour Creations Rocking Chair, Black"
26934,ES-2013-CD1192045-41609,2013-12-01,High,2013-12-05,Standard Class,OFF-BI-3247,"Avery Binder Covers, Durable"
36022,ES-2013-CD1192045-41609,2013-12-01,Medium,2013-12-07,Standard Class,OFF-BI-3723,"Cardinal Binder, Recycled"
40439,ES-2013-CD1192045-41609,2013-12-01,Medium,2013-12-07,Standard Class,OFF-AR-3452,"BIC Highlighters, Easy-Erase"
42867,ES-2013-CD1192045-41609,2013-12-01,Medium,2013-12-07,Standard Class,OFF-BI-2891,"Acco Binder Covers, Clear"
49743,ES-2013-CD1192045-41609,2013-12-01,High,2013-12-05,Standard Class,OFF-LA-4687,"Hon Round Labels, Alphabetical"


High priority → Dec 5
Medium priority → Dec 7

That is suspiciously systematic. It may indicate that priority is associated with the shipment/line, rather than being an erroneous value.

In [64]:
priority_shipdate = (valid_orders_data.groupby(["Order Priority", "Ship Date"]).size().reset_index(name="Rows"))
priority_shipdate

,Order Priority,Ship Date,Rows
0,Critical,2012-01-03,2
1,Critical,2012-01-08,1
2,Critical,2012-01-10,10
3,Critical,2012-01-15,3
4,Critical,2012-01-16,5
...,...,...,...
4647,Medium,2016-01-03,24
4648,Medium,2016-01-04,48
4649,Medium,2016-01-05,15
4650,Medium,2016-01-06,7


In [65]:
for order_id in priority_conflicts.index:
    temp = valid_orders_data[valid_orders_data["Order ID"] == order_id]
    print("\nORDER:", order_id)
    print(temp[[
                "Order Priority",
                "Ship Date",
                "Ship Mode",
                "Product ID",
                "Product Name"
            ]].drop_duplicates().to_string(index=False))


ORDER: ES-2013-CD1192045-41609
Order Priority  Ship Date      Ship Mode  Product ID                           Product Name
          High 2013-12-05 Standard Class TEC-CO-6011         Sharp Wireless Fax, High-Speed
        Medium 2013-12-07 Standard Class OFF-ST-4057          Eldon File Cart, Single Width
        Medium 2013-12-07 Standard Class FUR-CH-4545 Harbour Creations Rocking Chair, Black
          High 2013-12-05 Standard Class OFF-BI-3247           Avery Binder Covers, Durable
        Medium 2013-12-07 Standard Class OFF-BI-3723              Cardinal Binder, Recycled
        Medium 2013-12-07 Standard Class OFF-AR-3452           BIC Highlighters, Easy-Erase
        Medium 2013-12-07 Standard Class OFF-BI-2891              Acco Binder Covers, Clear
          High 2013-12-05 Standard Class OFF-LA-4687         Hon Round Labels, Alphabetical

ORDER: ES-2015-BG1174045-42353
Order Priority  Ship Date      Ship Mode  Product ID                     Product Name
      Critical 2015-12

In [66]:
priority_date_check = (valid_orders_data.groupby("Order Priority")["Ship Date"].nunique())
priority_date_check

Order Priority
Critical    1001
High        1423
Low          767
Medium      1461
Name: Ship Date, dtype: int64

In [67]:
(valid_orders_data.groupby("Ship Date")["Order Priority"].nunique().sort_values(ascending=False).head(20))

Ship Date
2012-12-03    4
2013-02-16    4
2012-01-23    4
2012-01-21    4
2015-12-17    4
2012-09-11    4
2013-06-03    4
2015-12-16    4
2015-12-23    4
2015-12-24    4
2015-12-25    4
2015-12-26    4
2015-08-16    4
2012-10-16    4
2015-12-29    4
2013-05-03    4
2015-12-31    4
2016-01-01    4
2016-01-02    4
2015-11-17    4
Name: Order Priority, dtype: int64

The priority appears to be associated with the shipping grouping.

Therefore, we should investigate whether:

Priority + Ship Date + Ship Mode

In [68]:
shipment_pattern = (valid_orders_data.groupby(["Order ID", "Order Priority", "Ship Date", "Ship Mode"]).size().reset_index(name="Line_Count"))
shipment_pattern.head(20)

,Order ID,Order Priority,Ship Date,Ship Mode,Line_Count
0,AE-2012-PO8865138-41184,Medium,2012-10-06,Standard Class,2
1,AE-2014-EB4110138-41926,High,2014-10-14,Same Day,2
2,AE-2014-MY7380138-42004,High,2015-01-03,Second Class,2
3,AE-2015-GH4665138-42351,Medium,2015-12-19,Standard Class,6
4,AE-2015-JD5790138-42070,Medium,2015-03-11,Standard Class,1
5,AE-2015-PG8820138-42313,Critical,2015-11-08,First Class,1
6,AG-2012-AA6453-41020,High,2012-04-24,First Class,1
7,AG-2012-AC4203-40915,Medium,2012-01-11,Standard Class,1
8,AG-2012-AH2103-41133,Medium,2012-08-16,Standard Class,1
9,AG-2012-AJ7803-40978,Medium,2012-03-15,Standard Class,1


In [69]:
shipment_pattern[["Order ID", "Order Priority", "Ship Date", "Ship Mode"]].drop_duplicates()

,Order ID,Order Priority,Ship Date,Ship Mode
0,AE-2012-PO8865138-41184,Medium,2012-10-06,Standard Class
1,AE-2014-EB4110138-41926,High,2014-10-14,Same Day
2,AE-2014-MY7380138-42004,High,2015-01-03,Second Class
3,AE-2015-GH4665138-42351,Medium,2015-12-19,Standard Class
4,AE-2015-JD5790138-42070,Medium,2015-03-11,Standard Class
...,...,...,...,...
25725,ZA-2015-RC9960146-42257,High,2015-09-15,Standard Class
25726,ZA-2015-RP9390146-42099,Critical,2015-04-07,First Class
25727,ZA-2015-SM10005146-42241,High,2015-08-25,Same Day
25728,ZA-2015-SW10350146-42061,Medium,2015-03-03,Standard Class


In [70]:
shipment_groups = (valid_orders_data.groupby(["Order ID", "Order Priority", "Ship Date", "Ship Mode"]).ngroups)
print("Shipment-like groups:", shipment_groups)

Shipment-like groups: 25730


In [71]:
shipment_count_per_order = (shipment_pattern.groupby("Order ID").size())
split_orders = shipment_count_per_order[shipment_count_per_order > 1]
print("Orders with multiple shipment-like groups:")
print(split_orders)

Orders with multiple shipment-like groups:
Order ID
ES-2013-CD1192045-41609     2
ES-2013-JF1541545-41509     2
ES-2015-BG1174045-42353     2
ES-2015-TM21490139-42223    2
ES-2015-TP21415139-42251    2
IN-2013-DM133457-41306      2
IN-2015-RF1984058-42273     2
MX-2014-TA2138582-41991     2
MX-2015-CC1210082-42318     2
NI-2014-TC1153595-41823     2
SA-2013-JP5460110-41467     2
UP-2014-BM1575137-41905     2
US-2015-SB2029018-42111     2
dtype: int64


In [72]:
print("Number of split orders:", len(split_orders))

Number of split orders: 13


In [73]:
for order_id in split_orders.index:
    print("\nORDER:", order_id)
    print(shipment_pattern[shipment_pattern["Order ID"] == order_id].to_string(index=False))


ORDER: ES-2013-CD1192045-41609
               Order ID Order Priority  Ship Date      Ship Mode  Line_Count
ES-2013-CD1192045-41609           High 2013-12-05 Standard Class           3
ES-2013-CD1192045-41609         Medium 2013-12-07 Standard Class           5

ORDER: ES-2013-JF1541545-41509
               Order ID Order Priority  Ship Date      Ship Mode  Line_Count
ES-2013-JF1541545-41509         Medium 2013-08-27   Second Class           2
ES-2013-JF1541545-41509         Medium 2013-08-27 Standard Class           1

ORDER: ES-2015-BG1174045-42353
               Order ID Order Priority  Ship Date      Ship Mode  Line_Count
ES-2015-BG1174045-42353       Critical 2015-12-17   Second Class           4
ES-2015-BG1174045-42353         Medium 2015-12-21 Standard Class           2

ORDER: ES-2015-TM21490139-42223
                Order ID Order Priority  Ship Date      Ship Mode  Line_Count
ES-2015-TM21490139-42223           High 2015-08-12 Standard Class           3
ES-2015-TM21490139-422

In [74]:
# SHIPPING COST
shipping_cost_check = (valid_orders_data.groupby(["Order ID", "Order Priority", "Ship Date", "Ship Mode"])["Shipping Cost"].nunique(dropna=False))
cost_conflicts = shipping_cost_check[shipping_cost_check > 1]
print("Shipment-like groups with multiple shipping costs:")
print(len(cost_conflicts))

Shipment-like groups with multiple shipping costs:
12776


In [75]:
for group in cost_conflicts.index[:10]:
    order_id, priority, ship_date, ship_mode = group
    temp = valid_orders_data[
        (valid_orders_data["Order ID"] == order_id) &
        (valid_orders_data["Order Priority"] == priority) &
        (valid_orders_data["Ship Date"] == ship_date) &
        (valid_orders_data["Ship Mode"] == ship_mode)]
    print("\nORDER:", order_id)
    print("PRIORITY:", priority)
    print("SHIP DATE:", ship_date)
    print("SHIP MODE:", ship_mode)
    print(temp[["Product ID",
                "Product Name",
                "Quantity",
                "Sales",
                "Shipping Cost"]].to_string(index=False))


ORDER: AE-2012-PO8865138-41184
PRIORITY: Medium
SHIP DATE: 2012-10-06 00:00:00
SHIP MODE: Standard Class
 Product ID                   Product Name  Quantity  Sales  Shipping Cost
OFF-ST-4258 Fellowes File Cart, Industrial         2 82.674           5.69
TEC-MA-4190          Epson Calculator, Red         6 78.408           3.87

ORDER: AE-2014-EB4110138-41926
PRIORITY: High
SHIP DATE: 2014-10-14 00:00:00
SHIP MODE: Same Day
 Product ID                  Product Name  Quantity   Sales  Shipping Cost
FUR-BO-3647 Bush Stackable Bookrack, Pine         6 224.748          60.08
OFF-FA-2945  Accos Paper Clips, Bulk Pack         1   4.248           1.10

ORDER: AE-2014-MY7380138-42004
PRIORITY: High
SHIP DATE: 2015-01-03 00:00:00
SHIP MODE: Second Class
 Product ID                       Product Name  Quantity  Sales  Shipping Cost
OFF-ST-6249                Tenex Folders, Blue         1  6.966           1.75
OFF-SU-6166 Stiletto Letter Opener, High Speed         2 16.668           1.41

ORDER:

In [76]:
# Order Priority + Ship Date + Ship Mode
shipment_key_check = (valid_orders_data.groupby(["Order ID", "Order Priority", "Ship Date", "Ship Mode"]).size())
print("Total shipment-like groups:", len(shipment_key_check))
print("Largest group size:", shipment_key_check.max())

Total shipment-like groups: 25730
Largest group size: 14


In [77]:
duplicate_shipment_groups = (valid_orders_data.groupby(["Order ID", "Order Priority", "Ship Date", "Ship Mode"]).size())
print("Duplicate shipment-like groups:",(duplicate_shipment_groups > 1).sum())

Duplicate shipment-like groups: 12783


In [78]:
shipment_groups_per_order = (
    valid_orders_data.groupby("Order ID").apply(lambda x: x[["Order Priority", "Ship Date", "Ship Mode"]].drop_duplicates().shape[0]))
split_orders = shipment_groups_per_order[shipment_groups_per_order > 1]
print("Orders with multiple shipment groups:", len(split_orders))
print(split_orders)

Orders with multiple shipment groups: 13
Order ID
ES-2013-CD1192045-41609     2
ES-2013-JF1541545-41509     2
ES-2015-BG1174045-42353     2
ES-2015-TM21490139-42223    2
ES-2015-TP21415139-42251    2
IN-2013-DM133457-41306      2
IN-2015-RF1984058-42273     2
MX-2014-TA2138582-41991     2
MX-2015-CC1210082-42318     2
NI-2014-TC1153595-41823     2
SA-2013-JP5460110-41467     2
UP-2014-BM1575137-41905     2
US-2015-SB2029018-42111     2
dtype: int64


In [79]:
for order_id in split_orders.index:
    print("\nORDER:", order_id)
    temp = valid_orders_data[valid_orders_data["Order ID"] == order_id]
    print(temp[["Order ID",
                "Order Priority",
                "Ship Date",
                "Ship Mode",
                "Product ID",
                "Product Name",
                "Quantity",
                "Sales",
                "Shipping Cost"]].to_string(index=False))


ORDER: ES-2013-CD1192045-41609
               Order ID Order Priority  Ship Date      Ship Mode  Product ID                           Product Name  Quantity   Sales  Shipping Cost
ES-2013-CD1192045-41609           High 2013-12-05 Standard Class TEC-CO-6011         Sharp Wireless Fax, High-Speed         3 903.006         182.15
ES-2013-CD1192045-41609         Medium 2013-12-07 Standard Class OFF-ST-4057          Eldon File Cart, Single Width         6 693.198          44.40
ES-2013-CD1192045-41609         Medium 2013-12-07 Standard Class FUR-CH-4545 Harbour Creations Rocking Chair, Black         5 650.970          35.85
ES-2013-CD1192045-41609           High 2013-12-05 Standard Class OFF-BI-3247           Avery Binder Covers, Durable         4  50.280           7.03
ES-2013-CD1192045-41609         Medium 2013-12-07 Standard Class OFF-BI-3723              Cardinal Binder, Recycled         3  42.570           3.28
ES-2013-CD1192045-41609         Medium 2013-12-07 Standard Class OFF-AR-34

In [80]:
print("Rows:", len(valid_orders_data))
print("Unique Orders:",valid_orders_data["Order ID"].nunique())
print("Shipment Groups:",valid_orders_data[["Order ID", "Order Priority", "Ship Date", "Ship Mode"]].drop_duplicates().shape[0])
print("Orders with multiple shipment groups:",len(split_orders))

Rows: 51255
Unique Orders: 25717
Shipment Groups: 25730
Orders with multiple shipment groups: 13


In [81]:
print(shipment_groups_per_order.value_counts().sort_index())

1    25704
2       13
Name: count, dtype: int64


25,704 orders × 1 shipment group = 25,704


13 orders × 2 shipment groups   =     26
                                      ─────
                                      25,730 shipment groups

25,704 + 13 = 25,717 unique orders

25,717 unique orders
        +
13 additional shipment groups
        =
25,730 shipment groups

We are currently defining a shipment as:

Order_ID+ Order_Priority+ Ship_Date+ Ship_Mode

That is a derived business key, not something provided by the original dataset.

In [82]:
shipment_key = ["Order ID","Order Priority","Ship Date","Ship Mode"]
shipment_group_check = (valid_orders_data.groupby(shipment_key).size())
print("Shipment groups:", len(shipment_group_check))
print("Total order lines:", shipment_group_check.sum())
print("Largest shipment:", shipment_group_check.max())

Shipment groups: 25730
Total order lines: 51255
Largest shipment: 14


In [83]:
shipment_grouped_rows = (valid_orders_data.groupby(shipment_key).size().sum())
print("Original rows:", len(valid_orders_data))
print("Rows assigned to shipment groups:", shipment_grouped_rows)

Original rows: 51255
Rows assigned to shipment groups: 51255


# Shipment Table

In [84]:
shipment_key = ["Order ID", "Order Priority", "Ship Date", "Ship Mode"]
shipments = (valid_orders_data[shipment_key].drop_duplicates().reset_index(drop=True))
shipments["Shipment ID"] = range(1, len(shipments) + 1)
shipments = shipments[["Shipment ID", "Order ID", "Order Priority", "Ship Date", "Ship Mode"]]
shipments.head()

,Shipment ID,Order ID,Order Priority,Ship Date,Ship Mode
0,1,CA-2014-AB10015140-41954,High,2014-11-13,First Class
1,2,IN-2014-JR162107-41675,Critical,2014-02-07,Second Class
2,3,IN-2014-CR127307-41929,Medium,2014-10-18,First Class
3,4,ES-2014-KM1637548-41667,Medium,2014-01-30,First Class
4,5,SG-2014-RH9495111-41948,Critical,2014-11-06,Same Day


In [85]:
print("Shipment rows:", len(shipments))
print("Unique Shipment IDs:", shipments["Shipment ID"].nunique())
print("Unique Orders:", shipments["Order ID"].nunique())

Shipment rows: 25730
Unique Shipment IDs: 25730
Unique Orders: 25717


ORDER_LINE → SHIPMENT : by merging the shipment table back into the order-line data

In [86]:
order_lines = valid_orders_data.merge(shipments,on=["Order ID", "Order Priority", "Ship Date", "Ship Mode"],how="left")

In [87]:
print("Order lines:", len(order_lines))
print("Missing Shipment IDs:", order_lines["Shipment ID"].isna().sum())
print("Unique Shipment IDs used:", order_lines["Shipment ID"].nunique())

Order lines: 51255
Missing Shipment IDs: 0
Unique Shipment IDs used: 25730


In [88]:
# CHECKING FOR SHIPPING COST

In [89]:
shipping_line_check = (order_lines.groupby(["Shipment ID"])["Shipping Cost"].nunique())
print("Shipments with multiple shipping costs:",(shipping_line_check > 1).sum())
print("Total shipments:",shipping_line_check.shape[0])

Shipments with multiple shipping costs: 12776
Total shipments: 25730


In [90]:
product_shipping_check = (order_lines.groupby(["Shipment ID", "Product ID"])["Shipping Cost"].nunique())
print("Shipment-product combinations with multiple shipping costs:",(product_shipping_check > 1).sum())

Shipment-product combinations with multiple shipping costs: 41


In [91]:
problem_ship_product = (product_shipping_check[product_shipping_check > 1])
print("Problematic shipment-product combinations:",len(problem_ship_product))
problem_ship_product.head(10)

Problematic shipment-product combinations: 41


Shipment ID  Product ID 
111          OFF-PA-6430    2
429          OFF-BI-6374    2
479          OFF-AP-4508    2
917          OFF-AP-4963    2
1557         FUR-BO-5942    2
2485         OFF-BI-6371    2
2876         OFF-AR-5926    2
3259         OFF-LA-5402    2
3566         FUR-TA-3765    2
3648         OFF-EN-3656    2
Name: Shipping Cost, dtype: int64

In [92]:
problem_pairs = problem_ship_product.index
investigation = (order_lines.set_index(["Shipment ID", "Product ID"]).loc[problem_pairs].reset_index())
investigation[["Shipment ID",
        "Order ID",
        "Product ID",
        "Product Name",
        "Quantity",
        "Sales",
        "Shipping Cost"]].sort_values(["Shipment ID", "Product ID"])

,Shipment ID,Order ID,Product ID,Product Name,Quantity,Sales,Shipping Cost
0,111,CA-2014-AB10060140-41884,OFF-PA-6430,Xerox 1881,2,24.560,6.530
1,111,CA-2014-AB10060140-41884,OFF-PA-6430,Xerox 1881,4,49.120,6.220
2,429,IN-2015-AM107057-42047,OFF-BI-6374,"Wilson Jones Binder Covers, Clear",5,47.925,2.120
3,429,IN-2015-AM107057-42047,OFF-BI-6374,"Wilson Jones Binder Covers, Clear",1,9.585,1.860
4,479,MX-2013-AG1090082-41398,OFF-AP-4508,"Hamilton Beach Stove, White",9,3242.880,279.530
...,...,...,...,...,...,...,...
77,24569,CA-2013-SS20515140-41479,OFF-BI-4357,"GBC Prepunched Paper, 19-Hole, for Binding Sys...",9,135.090,9.600
78,25154,US-2015-KB1640518-42328,FUR-FU-3039,"Advantus Light Bulb, Erganomic",3,14.952,1.248
79,25154,US-2015-KB1640518-42328,FUR-FU-3039,"Advantus Light Bulb, Erganomic",3,14.952,1.246
80,25164,CA-2015-TS21205140-42171,OFF-FA-6129,Staples,8,76.608,5.050


In [93]:
same_transaction = (order_lines.groupby(["Shipment ID", "Product ID", "Quantity", "Sales"])["Shipping Cost"].nunique())
print("Same shipment + product + quantity + sales ""with different shipping costs:",(same_transaction > 1).sum())

Same shipment + product + quantity + sales with different shipping costs: 10


In [94]:
suspicious = (order_lines[order_lines.duplicated(subset=[
                "Shipment ID",
                "Product ID",
                "Quantity",
                "Sales"],keep=False)].sort_values(["Shipment ID", "Product ID"]))
suspicious[[
        "Shipment ID",
        "Order ID",
        "Product ID",
        "Product Name",
        "Quantity",
        "Sales",
        "Shipping Cost"]]

,Shipment ID,Order ID,Product ID,Product Name,Quantity,Sales,Shipping Cost
34072,2485,ES-2014-SR20740120-41804,OFF-BI-6371,"Wilson Jones 3-Hole Punch, Economy",3,83.970,3.880
36496,2485,ES-2014-SR20740120-41804,OFF-BI-6371,"Wilson Jones 3-Hole Punch, Economy",3,83.970,3.140
37075,2876,ES-2013-RF1984014-41515,OFF-AR-5926,"Sanford Pens, Fluorescent",3,37.080,2.980
43181,2876,ES-2013-RF1984014-41515,OFF-AR-5926,"Sanford Pens, Fluorescent",3,37.080,1.840
38577,3648,KE-2014-PC900069-41693,OFF-EN-3656,"Cameo Business Envelopes, Set of 50",1,20.490,2.580
39265,3648,KE-2014-PC900069-41693,OFF-EN-3656,"Cameo Business Envelopes, Set of 50",1,20.490,2.410
4488,3682,IN-2013-RM19375130-41583,FUR-CH-5450,"Office Star Steel Folding Chair, Adjustable",5,343.830,72.950
6047,3682,IN-2013-RM19375130-41583,FUR-CH-5450,"Office Star Steel Folding Chair, Adjustable",5,343.830,55.800
4812,3928,MX-2013-LS1720026-41611,FUR-CH-5775,"SAFCO Executive Leather Armchair, Red",2,615.240,68.727
6154,3928,MX-2013-LS1720026-41611,FUR-CH-5775,"SAFCO Executive Leather Armchair, Red",2,615.240,54.907


In [95]:
duplicate_line_check = order_lines.duplicated(subset=["Order ID","Product ID","Quantity","Sales","Shipping Cost"],keep=False)
print("Exact duplicate transaction lines:",duplicate_line_check.sum())

Exact duplicate transaction lines: 0


In [96]:
shipments.to_csv(r"C:\Users\hp\OneDrive\Desktop\Retail-Analytics-Platform\Data\ETL data\Shipment.xls", index=False)
print("shipments.csv created")

shipments.csv created


# Order Line Table

In [97]:
order_lines["Unit Price"] = (order_lines["Sales"] / order_lines["Quantity"])
product_price_check = (order_lines.groupby("Product ID")["Unit Price"].nunique())
print("Products with multiple unit prices:",(product_price_check > 1).sum())
print("Total products:",product_price_check.shape[0])

Products with multiple unit prices: 3558
Total products: 3788


In [98]:
price_variation = (order_lines.groupby("Product ID").agg(
        Unit_Prices=("Unit Price", "unique"),
        Min_Price=("Unit Price", "min"),
        Max_Price=("Unit Price", "max"),
        Transactions=("Unit Price", "size")))
price_variation[price_variation["Transactions"] > 1].head(10)

,Unit_Prices,Min_Price,Max_Price,Transactions
Product ID,,,,
FUR-BO-3174,"[72.29400000000001, 163.86639999999994]",72.294,163.8664,2
FUR-BO-3175,"[260.98, 208.78400000000002, 182.686, 221.833,...",130.490,260.9800,8
FUR-BO-3176,"[238.833, 224.784, 84.29400000000001, 224.7840...",84.294,238.8330,6
FUR-BO-3177,"[255.833, 90.29400000000005, 240.784, 204.6664...",90.294,255.8330,8
FUR-BO-3409,"[49.995, 67.99319999999999, 99.99, 99.99000000...",29.997,99.9900,8
FUR-BO-3615,"[89.98289999999999, 142.83, 128.547, 142.82999...",57.132,142.8300,16
FUR-BO-3616,"[97.25999999999999, 145.89000000000001, 94.828...",43.767,145.8900,20
FUR-BO-3617,"[144.72, 77.184, 96.48, 96.47999999999999, 130...",43.416,144.7200,12
FUR-BO-3618,"[143.37, 129.03300000000002, 71.685, 95.58, 12...",43.011,143.3700,19


In [99]:
product_id = (product_price_check[product_price_check > 1]).index[0]
order_lines[order_lines["Product ID"] == product_id][[
        "Order ID",
        "Order Date",
        "Product ID",
        "Product Name",
        "Quantity",
        "Sales",
        "Unit Price"]].sort_values("Order Date")

,Order ID,Order Date,Product ID,Product Name,Quantity,Sales,Unit Price
6650,CA-2014-BT11440140-41718,2014-03-20,FUR-BO-3174,"Atlantic Metals Mobile 2-Shelf Bookcases, Cust...",1,72.2940,72.2940
30232,CA-2015-JB16000140-42326,2015-11-18,FUR-BO-3174,"Atlantic Metals Mobile 2-Shelf Bookcases, Cust...",2,327.7328,163.8664


In [100]:
order_lines = order_lines.copy()
order_lines.insert(0,"Order Line ID",range(1, len(order_lines) + 1))

In [101]:
order_line_columns = ["Order Line ID","Order ID","Shipment ID","Product ID","Quantity","Sales","Shipping Cost" ]
order_lines = order_lines[order_line_columns]

In [102]:
print("Rows:", len(order_lines))
print("Unique Order Line IDs:", order_lines["Order Line ID"].nunique())
print("Missing Order IDs:", order_lines["Order ID"].isna().sum())
print("Missing Shipment IDs:", order_lines["Shipment ID"].isna().sum())
print("Missing Product IDs:", order_lines["Product ID"].isna().sum())

Rows: 51255
Unique Order Line IDs: 51255
Missing Order IDs: 0
Missing Shipment IDs: 0
Missing Product IDs: 0


In [103]:
print("Duplicate Order Line IDs:",order_lines["Order Line ID"].duplicated().sum())

Duplicate Order Line IDs: 0


In [104]:
print(order_lines.columns.tolist())

['Order Line ID', 'Order ID', 'Shipment ID', 'Product ID', 'Quantity', 'Sales', 'Shipping Cost']


In [105]:
print(valid_orders_data.columns.tolist())

['Row ID', 'Order ID', 'Order Date', 'Ship Date', 'Ship Mode', 'Customer ID', 'Customer Name', 'Segment', 'Postal Code', 'City', 'State', 'Country', 'Region', 'Market', 'Product ID', 'Category', 'Sub-Category', 'Product Name', 'Sales', 'Quantity', 'Discount', 'Profit', 'Shipping Cost', 'Order Priority']


In [106]:
order_lines["Profit"] = valid_orders_data["Profit"].values

In [121]:
order_lines["Discount"] = valid_orders_data["Discount"].values

In [122]:
print(order_lines.shape)
print(order_lines.columns.tolist())
print(order_lines.head())

(51255, 9)
['Order Line ID', 'Order ID', 'Shipment ID', 'Product ID', 'Quantity', 'Sales', 'Shipping Cost', 'Profit', 'Discount']
   Order Line ID                  Order ID  Shipment ID   Product ID  \
0              1  CA-2014-AB10015140-41954            1  TEC-PH-5816   
1              2    IN-2014-JR162107-41675            2  FUR-CH-5379   
2              3    IN-2014-CR127307-41929            3  TEC-PH-5356   
3              4   ES-2014-KM1637548-41667            4  TEC-PH-5267   
4              5   SG-2014-RH9495111-41948            5  TEC-CO-6011   

   Quantity     Sales  Shipping Cost    Profit  Discount  
0         2   221.980          40.77   62.1544       0.0  
1         9  3709.395         923.63 -288.7650       0.1  
2         9  5175.171         915.49  919.9710       0.1  
3         5  2892.510         910.16  -96.5400       0.1  
4         8  2832.960         903.04  311.5200       0.0  


In [124]:
order_lines.to_csv(r"C:\Users\hp\OneDrive\Desktop\Retail-Analytics-Platform\Data\ETL data\Order line.xls", index=False)
print("order_lines.csv created")

order_lines.csv created


# Order Table

In [111]:
order_check = valid_orders_data.groupby("Order ID").agg({"Order Date": "nunique","Customer ID": "nunique"})
print("Orders with multiple Order Dates:",(order_check["Order Date"] > 1).sum())
print("Orders with multiple Customer IDs:",(order_check["Customer ID"] > 1).sum())

Orders with multiple Order Dates: 0
Orders with multiple Customer IDs: 0


In [112]:
order_location_check = valid_orders_data.groupby("Order ID").agg({
    "Order Date": "nunique",
    "Customer ID": "nunique",
    "City": "nunique",
    "State": "nunique",
    "Postal Code": "nunique",
    "Country": "nunique",
    "Region": "nunique",
    "Market": "nunique"})
print("Orders with multiple Order Dates:",(order_location_check["Order Date"] > 1).sum())
print("Orders with multiple Customer IDs:",(order_location_check["Customer ID"] > 1).sum())
print("Orders with multiple Cities:",(order_location_check["City"] > 1).sum())
print("Orders with multiple States:",(order_location_check["State"] > 1).sum())
print("Orders with multiple Postal Codes:",(order_location_check["Postal Code"] > 1).sum())
print("Orders with multiple Countries:",(order_location_check["Country"] > 1).sum())
print("Orders with multiple Regions:",(order_location_check["Region"] > 1).sum())
print("Orders with multiple Markets:",(order_location_check["Market"] > 1).sum())

Orders with multiple Order Dates: 0
Orders with multiple Customer IDs: 0
Orders with multiple Cities: 14
Orders with multiple States: 10
Orders with multiple Postal Codes: 0
Orders with multiple Countries: 0
Orders with multiple Regions: 0
Orders with multiple Markets: 0


In [113]:
print("Duplicate Order IDs:",orders["Order ID"].duplicated().sum())

NameError: name 'orders' is not defined

In [ ]:
orders = (valid_orders_data[[
            "Order ID",
            "Order Date",
            "Customer ID",
            "City",
            "State",
            "Postal Code",
            "Country",
            "Region",
            "Market"]].drop_duplicates("Order ID").sort_values("Order ID").reset_index(drop=True))

In [114]:
print("Order rows:", len(orders))
print("Unique Order IDs:", orders["Order ID"].nunique())
print(orders.head())

NameError: name 'orders' is not defined

In [115]:
orders.to_csv(r"C:\Users\hp\OneDrive\Desktop\Retail-Analytics-Platform\Data\ETL data\Orders.xls", index=False)
print("orders.csv created successfully")

NameError: name 'orders' is not defined

# Returns Table

In [116]:
returns = pd.read_excel(r"C:\Users\hp\OneDrive\Desktop\Retail-Analytics-Platform\Data\raw.xlsx", sheet_name="Returns", engine="openpyxl")

In [117]:
returns.to_csv(r"C:\Users\hp\OneDrive\Desktop\Retail-Analytics-Platform\Data\ETL data\Returns.xls", index=False)

In [118]:
order_lines.columns.tolist()

['Order Line ID',
 'Order ID',
 'Shipment ID',
 'Product ID',
 'Quantity',
 'Sales',
 'Shipping Cost',
 'Profit']